# Deep Past Challenge — Akkadian → English Translation

This notebook loads a fine-tuned ByT5-base model and generates English translations from Akkadian transliterations.

**Setup:** Add the `byt5-akkadian-checkpoint` dataset to this notebook (contains the trained model).

In [ ]:
import os
import re
import unicodedata
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────
OUTPUT_CSV = '/kaggle/working/submission.csv'

BATCH_SIZE = 16
NUM_BEAMS = 4
MAX_LENGTH = 512

# Auto-detect model path by scanning all /kaggle/input/ for config.json
import glob
INPUT_ROOT = '/kaggle/input'
matches = glob.glob(os.path.join(INPUT_ROOT, '**', 'config.json'), recursive=True)
if not matches:
    raise FileNotFoundError(f"No config.json found under {INPUT_ROOT}")
MODEL_PATH = os.path.dirname(matches[0])

# Auto-detect test.csv path
test_matches = glob.glob(os.path.join(INPUT_ROOT, '**', 'test.csv'), recursive=True)
if not test_matches:
    raise FileNotFoundError(f"No test.csv found under {INPUT_ROOT}")
TEST_CSV = test_matches[0]

print(f"Model path: {MODEL_PATH}")
print(f"Model files: {os.listdir(MODEL_PATH)}")
print(f"Test CSV: {TEST_CSV}")

In [ ]:
# ── Preprocessing ──────────────────────────────────────────────────────
_SUBSCRIPT_MAP = str.maketrans('\u2080\u2081\u2082\u2083\u2084\u2085\u2086\u2087\u2088\u2089', '0123456789')
_VOWEL_ACCENTS = [
    (re.compile(r'a2\b'), '\u00e1'), (re.compile(r'a3\b'), '\u00e0'),
    (re.compile(r'e2\b'), '\u00e9'), (re.compile(r'e3\b'), '\u00e8'),
    (re.compile(r'i2\b'), '\u00ed'), (re.compile(r'i3\b'), '\u00ec'),
    (re.compile(r'u2\b'), '\u00fa'), (re.compile(r'u3\b'), '\u00f9'),
]

def clean_transliteration(text):
    if not isinstance(text, str) or not text.strip():
        return ''
    text = unicodedata.normalize('NFKC', text)
    text = text.replace('\u1e2a', 'H').replace('\u1e2b', 'h')
    text = text.translate(_SUBSCRIPT_MAP)
    text = re.sub(r'\[(?:x\s*)+\]', '<gap>', text)
    text = re.sub(r'\[\.{2,}\]', '<gap>', text)
    text = re.sub(r'\.{3,}', '<gap>', text)
    text = re.sub(r'\u2026', '<gap>', text)
    text = re.sub(r'\[broken\]', '<gap>', text, flags=re.IGNORECASE)
    text = re.sub(r'[\u02f9\u2e22\u02fa\u2e23]', '', text)
    text = re.sub(r'\[([^\]]*)\]', r'\1', text)
    text = re.sub(r'<(?!gap>)([^>]*)>', r'\1', text)
    text = re.sub(r'[!?*#]', '', text)
    text = text.replace('sz', '\u0161').replace('SZ', '\u0160')
    text = re.sub(r's,', '\u1e63', text)
    text = re.sub(r't,', '\u1e6d', text)
    for p, r in _VOWEL_ACCENTS:
        text = p.sub(r, text)
    text = re.sub(r'(<gap>\s*){2,}', '<gap> ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Quick test
print(clean_transliteration('um-ma kà-ru-um kà-ni-ia-ma'))

In [ ]:
# ── Load Model ─────────────────────────────────────────────────────────
print('Loading model and tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH, local_files_only=True).to(DEVICE)
model.eval()
print('Model loaded successfully.')

In [ ]:
# ── Load and Prepare Test Data ──────────────────────────────────────────
test_df = pd.read_csv(TEST_CSV)
print(f'Test samples: {len(test_df)}')
print(f'Columns: {list(test_df.columns)}')
print(test_df.head())

# Find the transliteration column
translit_col = next((c for c in test_df.columns if 'translit' in c.lower()), test_df.columns[-1])
id_col = 'id' if 'id' in test_df.columns else test_df.columns[0]
print(f'\nUsing transliteration column: {translit_col}')
print(f'Using ID column: {id_col}')

# Clean transliterations
test_df['clean'] = test_df[translit_col].apply(clean_transliteration)
print(f'\nSample cleaned:')
print(test_df[['clean']].head())

In [ ]:
# ── Run Inference ───────────────────────────────────────────────────────
texts = test_df['clean'].tolist()
predictions = []

for i in range(0, len(texts), BATCH_SIZE):
    batch = [t if t else '<gap>' for t in texts[i:i + BATCH_SIZE]]
    inputs = tokenizer(
        batch, max_length=MAX_LENGTH, truncation=True,
        padding=True, return_tensors='pt'
    ).to(DEVICE)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            num_beams=NUM_BEAMS,
            max_length=MAX_LENGTH,
            early_stopping=True,
        )
    
    decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    predictions.extend(decoded)
    
    done = min(i + BATCH_SIZE, len(texts))
    if done % 50 == 0 or done == len(texts):
        print(f'  Processed {done}/{len(texts)} samples')

print(f'\nGenerated {len(predictions)} predictions')

In [ ]:
# ── Create Submission ───────────────────────────────────────────────────
submission = pd.DataFrame({
    'id': test_df[id_col],
    'translation': predictions,
})
submission['translation'] = submission['translation'].fillna('')
submission.to_csv(OUTPUT_CSV, index=False)

print(f'Submission saved to: {OUTPUT_CSV}')
print(f'Shape: {submission.shape}')
print(f'\nFirst 5 predictions:')
for _, row in submission.head(5).iterrows():
    print(f"  [{row['id']}]: {row['translation'][:120]}")

# Verify format against sample
sample_matches = glob.glob(os.path.join('/kaggle/input', '**', 'sample_submission.csv'), recursive=True)
sample_path = sample_matches[0] if sample_matches else None
if sample_path and os.path.exists(sample_path):
    sample = pd.read_csv(sample_path)
    print(f'\nFormat check:')
    print(f'  Expected columns: {list(sample.columns)}')
    print(f'  Got columns:      {list(submission.columns)}')
    print(f'  Expected rows:    {len(sample)}')
    print(f'  Got rows:         {len(submission)}')
    match = list(sample.columns) == list(submission.columns)
    print(f'  Columns match: {"YES" if match else "NO"}')